# LLM topic labeling

Generate three Portuguese label suggestions for each final topic. Gold questions are explicitly excluded. Run the input preparation cells first and inspect `topic_labeling_inputs.csv` before making API calls.

In [14]:
import hashlib
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
HIERARCHY_DIR = PROJECT_DIR / 'results/hierarchical_topics/nn30_mcs20_seed2024'
ASSIGNMENTS_PATH = HIERARCHY_DIR / 'unique_questions_with_fine_topics.csv'
TOPICS_PATH = HIERARCHY_DIR / 'fine_topic_descriptions.csv'
GOLD_PATH = PROJECT_DIR / 'data/files/queries.txt'
EMBEDDINGS_PATH = PROJECT_DIR / 'results/hierarchical_topics/embeddings_all_questions_google_embeddinggemma-300m.npy'
OUTPUT_DIR = PROJECT_DIR / 'results/topic_labeling'
MODEL = 'gpt-5.4-nano'
MAX_EXAMPLES = 50
CENTRAL_EXAMPLES = 25
MMR_DIVERSITY = 0.5
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [15]:
if not ASSIGNMENTS_PATH.exists():
    raise FileNotFoundError('Execute hierarchical_topics.ipynb antes deste notebook.')

questions = pd.read_csv(ASSIGNMENTS_PATH)
topics = pd.read_csv(TOPICS_PATH)
gold_questions = {line.strip() for line in GOLD_PATH.read_text(encoding='utf-8').splitlines() if line.strip()}
questions = questions[(questions['fine_topic_id'] != -1) & ~questions['perguntas'].isin(gold_questions)].drop_duplicates('perguntas').reset_index(drop=True)
all_embeddings = np.load(EMBEDDINGS_PATH)

source = pd.read_csv(PROJECT_DIR / 'data/perguntas/relevant_question_extraction_gpt-5-4-mini_high_flex.csv')
source = source[source['perguntas'].ne('[]')].copy()
import ast
source['perguntas'] = source['perguntas'].apply(lambda value: ast.literal_eval(value) if isinstance(value, str) else value)
all_occurrences = source.explode('perguntas')['perguntas'].dropna().astype(str).reset_index(drop=True)
first_occurrence = ~all_occurrences.duplicated()
embedding_by_question = dict(zip(all_occurrences[first_occurrence], np.flatnonzero(first_occurrence)))
if len(all_embeddings) != len(all_occurrences):
    raise ValueError('O número de embeddings difere do número de ocorrências das perguntas.')
questions['embedding_index'] = questions['perguntas'].map(embedding_by_question)
if questions['embedding_index'].isna().any():
    raise ValueError('As perguntas e os embeddings não estão alinhados.')

def select_examples(frame):
    indices = frame['embedding_index'].astype(int).to_numpy()
    vectors = all_embeddings[indices]
    vectors = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)
    centroid = vectors.mean(axis=0)
    centroid /= np.linalg.norm(centroid)
    relevance = vectors @ centroid
    if len(frame) <= MAX_EXAMPLES:
        selected = np.argsort(relevance)[::-1]
    else:
        central = list(np.argsort(relevance)[::-1][:CENTRAL_EXAMPLES])
        selected = central.copy()
        candidates = set(range(len(frame))) - set(selected)
        while len(selected) < MAX_EXAMPLES and candidates:
            candidate_list = np.asarray(sorted(candidates))
            redundancy = (vectors[candidate_list] @ vectors[selected].T).max(axis=1)
            score = MMR_DIVERSITY * relevance[candidate_list] - (1 - MMR_DIVERSITY) * redundancy
            chosen = int(candidate_list[score.argmax()])
            selected.append(chosen)
            candidates.remove(chosen)
    return frame.iloc[selected[:MAX_EXAMPLES]]['perguntas'].tolist()

topic_words = topics.set_index('Topic')['Representation'].to_dict()
rows = []
for topic_id, frame in questions.groupby('fine_topic_id'):
    examples = select_examples(frame)
    rows.append({
        'topic_id': int(topic_id),
        'topic_size': len(frame),
        'top_words': topic_words.get(topic_id, ''),
        'example_count': len(examples),
        'examples_json': json.dumps(examples, ensure_ascii=False),
        'examples_sha256': hashlib.sha256('\n'.join(examples).encode()).hexdigest(),
    })
labeling_inputs = pd.DataFrame(rows).sort_values('topic_id').reset_index(drop=True)
labeling_inputs.to_csv(OUTPUT_DIR / 'topic_labeling_inputs.csv', index=False)
selected_question_rows = []
for row in labeling_inputs.itertuples(index=False):
    for example_order, question in enumerate(json.loads(row.examples_json), start=1):
        selected_question_rows.append({
            'topic_id': row.topic_id,
            'topic_size': row.topic_size,
            'top_words': row.top_words,
            'example_order': example_order,
            'selected_question': question,
        })
selected_questions = pd.DataFrame(selected_question_rows)
selected_questions.to_csv(OUTPUT_DIR / 'topic_labeling_selected_questions.csv', index=False)
display(labeling_inputs.head())
selected_questions.head()

,topic_id,topic_size,top_words,example_count,examples_json,examples_sha256
0,0,1267,"['maternidade', 'salário', 'auxílio', 'direito...",50,"[""Uma gestante de 6 meses que vai começar a co...",795cc27451a5e5d876110a84821e4a51bf3fdbc27dca0a...
1,1,360,"['anticoncepcional', 'pílula', 'seguinte', 'to...",50,"[""Tive relação no 9º dia de uso do anticoncepc...",db7aec3be44a3156f573d6932e95f0af11e77144ce7fcd...
2,2,361,"['progesterona', 'via', 'oral', 'vaginal', 'us...",50,"[""A progesterona precisa ser usada por via ora...",6f9d6e276d22470110c3c3d4944132e8a46451c7437589...
3,3,308,"['ela', 'grávida', 'está', 'pessoa', 'menciona...",50,"[""Será que ela está mesmo grávida?"", ""Ela esta...",dde0a416ba0d9613e3737e75c8b749f95baf91714d4648...
4,4,264,"['nome', 'for', 'nomes', 'vocês', 'chamar', 'q...",50,"[""Qual nome seria escolhido se o bebê for meni...",b34d3d757522e6ee02fea3734a2c3485fcec3c4d3cf5df...


,topic_id,topic_size,top_words,example_order,selected_question
0,0,1267,"['maternidade', 'salário', 'auxílio', 'direito...",1,Uma gestante de 6 meses que vai começar a cont...
1,0,1267,"['maternidade', 'salário', 'auxílio', 'direito...",2,É necessário contribuir por mais um mês para t...
2,0,1267,"['maternidade', 'salário', 'auxílio', 'direito...",3,Com quantos meses de contribuição é possível t...
3,0,1267,"['maternidade', 'salário', 'auxílio', 'direito...",4,"No meu caso, preciso aguardar minha filha nasc..."
4,0,1267,"['maternidade', 'salário', 'auxílio', 'direito...",5,"Se o bebê já nasceu, ainda é possível receber ..."


In [ ]:
from openai import OpenAI
from pydantic import BaseModel, Field

OPENAI_API_KEY = 'COLE_SUA_CHAVE_AQUI'
if OPENAI_API_KEY == 'COLE_SUA_CHAVE_AQUI':
    raise ValueError('Substitua COLE_SUA_CHAVE_AQUI pela sua chave.')

class TopicNames(BaseModel):
    primary_name: str
    alternative_name_1: str
    alternative_name_2: str

SYSTEM_PROMPT = """Você nomeia tópicos formados por perguntas sobre gravidez e saúde materna.

Primeiro identifique internamente um único tema central que represente a maioria das perguntas. Não apresente essa análise.

Depois crie um nome principal para esse tema e duas reformulações do nome principal.
As três opções devem nomear exatamente o mesmo assunto e possuir o mesmo nível de abrangência.
As alternativas não podem representar subtemas exemplos ou aspectos diferentes do nome principal.
Não use as três opções para cobrir partes diferentes do cluster.

Cada nome deve ser um sintagma nominal em português com 2 a 7 palavras e funcionar como rótulo de uma categoria.
Não escreva perguntas instruções ou títulos de tutorial.
Não use pontuação nem enumere elementos.
Não use termos normativos como correto ideal ou melhor.

A saída deve conter somente o nome principal e suas duas paráfrases."""

client = OpenAI(api_key=OPENAI_API_KEY)
checkpoint_path = OUTPUT_DIR / 'topic_label_suggestions_checkpoint.csv'
if checkpoint_path.exists():
    completed = pd.read_csv(checkpoint_path)
else:
    completed = pd.DataFrame(columns=['topic_id', 'suggestion_1', 'suggestion_2', 'suggestion_3', 'model', 'response_id', 'examples_sha256', 'error'])

completed_ids = set(completed.loc[completed['error'].fillna('').eq(''), 'topic_id'].astype(int)) if len(completed) else set()
new_rows = []

# Para testar tópico específico 
# TOPIC_ID_TO_TEST = 20
# labeling_inputs_to_process = labeling_inputs[
#     labeling_inputs['topic_id'] == TOPIC_ID_TO_TEST
# ].copy()

# for row in labeling_inputs_to_process.itertuples(index=False):

for row in labeling_inputs.itertuples(index=False): # comentar se for testar apenas um tópico específico
    if row.topic_id in completed_ids:
        continue
    examples = json.loads(row.examples_json)
    prompt = f'Tópico {row.topic_id}\nPalavras principais: {row.top_words}\nPerguntas:\n' + '\n'.join(f'{i + 1}. {question}' for i, question in enumerate(examples))
    try:
        response = client.responses.parse(
            model=MODEL,
            input=[{'role': 'system', 'content': SYSTEM_PROMPT}, {'role': 'user', 'content': prompt}],
            text_format=TopicNames,
        )
        names = response.output_parsed
        result = {'topic_id': row.topic_id, 'suggestion_1': names.primary_name, 'suggestion_2': names.alternative_name_1, 'suggestion_3': names.alternative_name_2, 'model': MODEL, 'response_id': response.id, 'examples_sha256': row.examples_sha256, 'error': ''}
    except Exception as error:
        result = {'topic_id': row.topic_id, 'suggestion_1': '', 'suggestion_2': '', 'suggestion_3': '', 'model': MODEL, 'response_id': '', 'examples_sha256': row.examples_sha256, 'error': repr(error)}
    new_rows.append(result)
    pd.concat([completed, pd.DataFrame(new_rows)], ignore_index=True).drop_duplicates('topic_id', keep='last').to_csv(checkpoint_path, index=False)

suggestions = pd.read_csv(checkpoint_path)
suggestions = labeling_inputs.drop(columns=['examples_json']).merge(suggestions, on=['topic_id', 'examples_sha256'], how='left')
suggestions['selected_name'] = ''
suggestions['decision'] = ''
suggestions.to_csv(OUTPUT_DIR / 'topic_label_suggestions.csv', index=False)
suggestions.head()